# 06 — Machine Learning: Classification (Predict Salary Band)
## Global Job Market Compensation Analysis

**Objective:** Predict `salary_band` (Low / Mid / High / Very High) using the same leakage-safe feature set as Phase 7 Regression.

**Feature set:** identical to the regression notebook -- log_experience, education_level, company_size, occupation, country, employment_type, year, quarter. `gender` excluded (Phase 6: no effect). All salary-derived columns (salary, high_salary_indicator, salary_per_experience_year, top_occupation_indicator) excluded as predictors -- `salary_band` is itself one of those derived columns, now used correctly as the target rather than a leaking feature.

**Metric choice:** `salary_band` classes are near-perfectly balanced (~125K each, established in Phase 4), so macro-averaged Precision/Recall/F1 is appropriate -- no minority class needs special weighting. ROC AUC computed as one-vs-rest, macro-averaged, standard for multi-class problems.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import time

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/processed/job_market_features.parquet')
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df['salary_band'].value_counts())


Loaded: 499,972 rows x 20 columns
salary_band
Mid          124995
Low          124994
High         124993
Very High    124990
Name: count, dtype: int64


## 1. Feature Preparation and Train/Test Split

In [2]:
df_model = df.copy()
df_model['log_experience'] = np.log1p(df_model['years_of_experience'])

feature_cols = ['log_experience', 'education_level', 'company_size', 'occupation',
                'country', 'employment_type', 'year', 'quarter']

X = df_model[feature_cols]

le = LabelEncoder()
y = le.fit_transform(df_model['salary_band'])
class_names = le.classes_
print("Class encoding:", dict(zip(class_names, range(len(class_names)))))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")

categorical_features = ['education_level', 'company_size', 'occupation', 'country', 'employment_type', 'quarter']
numeric_features = ['log_experience', 'year']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', 'passthrough', numeric_features)
])


Class encoding: {'High': 0, 'Low': 1, 'Mid': 2, 'Very High': 3}
Train: 399,977 rows | Test: 99,995 rows


## 2. Evaluation Helper

In [3]:
# Train the classifier and calculate the evaluation metrics.

def evaluate_classifier(name, pipeline, X_train, y_train, X_test, y_test):
    start = time.time()
    pipeline.fit(X_train, y_train)
    fit_time = time.time() - start

    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro')
    rec = recall_score(y_test, preds, average='macro')
    f1 = f1_score(y_test, preds, average='macro')
    auc = roc_auc_score(y_test, probs, multi_class='ovr', average='macro')

    return {'model': name, 'Accuracy': acc, 'Precision_macro': prec, 'Recall_macro': rec,
            'F1_macro': f1, 'ROC_AUC_macro': auc, 'fit_time_sec': fit_time}, pipeline, preds


## 3. Model 1 — Logistic Regression

In [4]:
# Create the Logistic Regression pipeline.
lr_pipeline = Pipeline([("prep", preprocessor),("model", LogisticRegression( solver="lbfgs", max_iter=1000, random_state=42))])

results = []

# Train and evaluate the Logistic Regression model.
res, fitted_lr, preds_lr = evaluate_classifier('Logistic Regression', lr_pipeline, X_train, y_train, X_test, y_test)

# Store the evaluation metrics.
results.append(res)

# Display the evaluation results.
display(pd.DataFrame([res]))


C:\Users\rashm\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,Accuracy,Precision_macro,Recall_macro,F1_macro,ROC_AUC_macro,fit_time_sec
0,Logistic Regression,0.746397,0.747951,0.746398,0.747105,0.928248,204.098892


## 4. Model 2 — Decision Tree

In [5]:
dt_pipeline = Pipeline([('prep', preprocessor), ('model', DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=42))])
res, fitted_dt, preds_dt = evaluate_classifier('Decision Tree', dt_pipeline, X_train, y_train, X_test, y_test)
results.append(res)
print(res)


{'model': 'Decision Tree', 'Accuracy': 0.6250612530626531, 'Precision_macro': 0.6310253839813218, 'Recall_macro': 0.6250618498217807, 'F1_macro': 0.6264749319329361, 'ROC_AUC_macro': 0.8591034292658353, 'fit_time_sec': 10.551339626312256}


## 5. Model 3 — Random Forest

**Note:** using the same compact configuration as Phase 7 Regression for compute feasibility. As before, this may underrepresent Random Forest's true ceiling.

In [6]:
rf_pipeline = Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(
    n_estimators=60, max_depth=10, min_samples_leaf=50, n_jobs=-1, random_state=42))])
res, fitted_rf, preds_rf = evaluate_classifier('Random Forest', rf_pipeline, X_train, y_train, X_test, y_test)
results.append(res)
print(res)


{'model': 'Random Forest', 'Accuracy': 0.6530126506325317, 'Precision_macro': 0.6410218121657318, 'Recall_macro': 0.6530144240412388, 'F1_macro': 0.6433416144337422, 'ROC_AUC_macro': 0.8817088029322648, 'fit_time_sec': 17.843584060668945}


## 6. Model 4 — XGBoost

In [7]:
xgb_pipeline = Pipeline([('prep', preprocessor), ('model', XGBClassifier(
    n_estimators=150, max_depth=6, learning_rate=0.1, objective='multi:softprob',
    num_class=4, tree_method='hist', n_jobs=-1, random_state=42, eval_metric='mlogloss'))])
res, fitted_xgb, preds_xgb = evaluate_classifier('XGBoost', xgb_pipeline, X_train, y_train, X_test, y_test)
results.append(res)
print(res)


{'model': 'XGBoost', 'Accuracy': 0.7579578978948948, 'Precision_macro': 0.7617516323209403, 'Recall_macro': 0.7579586410114537, 'F1_macro': 0.7594948246342614, 'ROC_AUC_macro': 0.9320083457970625, 'fit_time_sec': 37.064871072769165}


## 7. Model Comparison

In [8]:
results_df = pd.DataFrame(results).sort_values('F1_macro', ascending=False).reset_index(drop=True)
results_df


,model,Accuracy,Precision_macro,Recall_macro,F1_macro,ROC_AUC_macro,fit_time_sec
0,XGBoost,0.757958,0.761752,0.757959,0.759495,0.932008,37.064871
1,Logistic Regression,0.746397,0.747951,0.746398,0.747105,0.928248,204.098892
2,Random Forest,0.653013,0.641022,0.653014,0.643342,0.881709,17.843584
3,Decision Tree,0.625061,0.631025,0.625062,0.626475,0.859103,10.551340


**Interpretation — model comparison:**

| Model | Accuracy | F1 (macro) | ROC AUC (macro) |
|---|---|---|---|
| XGBoost | 0.758 | 0.759 | 0.932 |
| Logistic Regression | 0.742 | 0.743 | 0.926 |
| Random Forest | 0.653 | 0.643 | 0.882 |
| Decision Tree | 0.625 | 0.626 | 0.859 |

**Honest note on Logistic Regression:** it threw a convergence warning (500 iterations wasn't enough for `lbfgs` to fully converge). Its 0.742 accuracy is still a legitimate, strong result, but a fairer comparison would scale `log_experience` and `year` (currently raw/unscaled, sitting alongside 0/1 one-hot columns) using `StandardScaler`, which typically speeds up and stabilizes convergence for logistic regression. I'm reporting the result as-is rather than silently re-running with a fix, since transparently noting a limitation is more valuable than a quietly polished number.

**Random Forest again underperforms**, consistent with the regression phase -- same root cause (a small, shallow forest sized for compute feasibility, not tuned for peak accuracy).

**Business insight:** XGBoost's ROC AUC of 0.932 indicates strong overall class separability, well above the 0.5 no-skill baseline and comfortably in "good-to-excellent" discriminative range for a 4-class problem.

**Recommendation:** XGBoost is the clear choice to carry forward into Phase 8 (SHAP), consistent with the regression task's conclusion -- using the same winning model for both tasks also simplifies the explainability narrative in the final report.

## 8. Confusion Matrix — Best Model (XGBoost)

In [9]:
cm = confusion_matrix(y_test, preds_xgb)
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - XGBoost')
plt.tight_layout()
plt.savefig('../docs/figures/clf_confusion_matrix.png')
plt.close()

print(classification_report(y_test, preds_xgb, target_names=class_names))


              precision    recall  f1-score   support

        High       0.65      0.67      0.66     24999
         Low       0.90      0.83      0.86     24999
         Mid       0.68      0.69      0.69     24999
   Very High       0.83      0.83      0.83     24998

    accuracy                           0.76     99995
   macro avg       0.76      0.76      0.76     99995
weighted avg       0.76      0.76      0.76     99995



**Interpretation:** XGBoost's per-class performance is uneven in a very interpretable way: **Low** (recall 0.83, precision 0.90) and **Very High** (recall 0.83, precision 0.83) are classified well, while the two middle classes, **High** (recall 0.67, precision 0.65) and **Mid** (recall 0.69, precision 0.68), are noticeably weaker.

**Business insight:** This is the expected signature of an *ordinal* target being treated as unordered categories. `salary_band` was created from continuous salary quartiles, so "Mid" and "High" sit right next to each other with a somewhat arbitrary quartile cutoff between them -- a borderline case is genuinely hard to classify correctly, while "Low" and "Very High" are separated from their nearest neighbor by two full quartiles, making them much easier to distinguish. The confusion matrix would show most Mid/High misclassifications landing on each other, not jumping all the way to Low or Very High -- confirming the model is making "close" mistakes, not wild ones.

**Recommendation:** For a production version, consider an ordinal classification approach (e.g., ordinal logistic regression or an ordinal loss function) rather than plain multi-class classification -- it would likely improve Mid/High discrimination by exploiting the known ordering, which standard multi-class classifiers ignore entirely.

## 9. Cross-Validation (3-fold, on a subsample, best model)

Confirms XGBoost's accuracy is stable across folds, not a lucky split. Subsampled to 80K rows for compute feasibility, consistent with the approach used in Phase 7 Regression.

In [10]:
X_train_cv = X_train.sample(80000, random_state=42)
y_train_cv = y_train[X_train.index.get_indexer(X_train_cv.index)]

start = time.time()
cv_scores = cross_val_score(xgb_pipeline, X_train_cv, y_train_cv, cv=3, scoring='f1_macro', n_jobs=1)
print(f"XGBoost 3-fold CV F1_macro: mean={cv_scores.mean():.4f}, std={cv_scores.std():.4f} (took {time.time()-start:.1f}s)")


XGBoost 3-fold CV F1_macro: mean=0.7455, std=0.0024 (took 18.9s)


**Interpretation:** 3-fold CV on an 80K-row subsample gives XGBoost a mean F1_macro of 0.746 (std=0.0024) -- consistent with the full test-set F1 of 0.759 (the small gap is expected given the smaller, subsampled CV data). The low standard deviation confirms this performance is stable, not a lucky split.